# 02 — Data Cleaning and Integration (revised v3)

This version fixes the LSOA supply identity-check filter so that it selects
the rows with existing provision rather than extracting the Boolean column.


## 0. Setup and cleaning parameters


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

BASE = Path("/Users/alexia/Documents/CASA/Dissertation")
OUTPUT_DIR = BASE / "05_processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "ts045": BASE / "03_data/demand/census2021/TS045_car_van_availability_London_LSOA_2021.csv",
    "iod2025": BASE / "03_data/demand/IoD2025.csv",
    "evse_location": BASE / "03_data/restricted/evse_location.csv",
    "charging_activity": BASE / "03_data/restricted/charging_activity_sep.csv",
    "lsoa_boundaries": BASE / "03_data/demand/spatial/LSOA_2021_EW_BGC_V5.shp",
}

# Study-specific assumptions and diagnostics
TOP_CODED_VEHICLES = 3.0          # conservative value assigned to Census category "3 or more"
WINDOW_START = pd.Timestamp("2025-09-01 00:00:00")
WINDOW_END = pd.Timestamp("2025-10-01 00:00:00")
WINDOW_MINUTES = (WINDOW_END - WINDOW_START).total_seconds() / 60
SHORT_SESSION_THRESHOLD_MIN = 1.0  # flagged for sensitivity analysis, retained in core data
LONG_SESSION_THRESHOLD_MIN = 24 * 60  # flagged for review, not automatically removed
NEAREST_MATCH_MAX_METRES = 100.0

for name, path in PATHS.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

print("Output directory:", OUTPUT_DIR)
print("Observation window:", WINDOW_START, "to", WINDOW_END,
      f"({WINDOW_MINUTES:,.0f} minutes)")


Output directory: /Users/alexia/Documents/CASA/Dissertation/05_processed
Observation window: 2025-09-01 00:00:00 to 2025-10-01 00:00:00 (43,200 minutes)


In [2]:
def require_columns(df: pd.DataFrame, required: list[str], dataset_name: str) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f"{dataset_name}: missing required columns: {missing}")


def first_non_null(series: pd.Series):
    values = series.dropna()
    return values.iloc[0] if not values.empty else np.nan


def mode_or_first(series: pd.Series):
    values = series.dropna()
    if values.empty:
        return np.nan
    modes = values.mode()
    return modes.iloc[0] if not modes.empty else values.iloc[0]


def assert_single_mapping(df: pd.DataFrame, group_col: str, value_cols: list[str], label: str) -> None:
    """Ensure each group maps to at most one non-null value in each structural field."""
    problems = {}
    grouped = df.groupby(group_col, dropna=False)
    for col in value_cols:
        counts = grouped[col].nunique(dropna=True)
        n_bad = int((counts > 1).sum())
        if n_bad:
            problems[col] = n_bad
    if problems:
        raise ValueError(f"{label}: inconsistent mappings detected: {problems}")


def merge_intervals_minutes(
    starts: pd.Series,
    ends: pd.Series,
    window_start: pd.Timestamp,
    window_end: pd.Timestamp,
) -> float:
    """Merge overlapping intervals and return occupied minutes clipped to the study window."""
    intervals = sorted(zip(starts, ends), key=lambda pair: pair[0])
    merged = []
    for start, end in intervals:
        if pd.isna(start) or pd.isna(end) or end <= start:
            continue
        start = max(start, window_start)
        end = min(end, window_end)
        if end <= start:
            continue
        if not merged or start > merged[-1][1]:
            merged.append([start, end])
        else:
            merged[-1][1] = max(merged[-1][1], end)
    return sum((end - start).total_seconds() / 60 for start, end in merged)


## 1. Reload and standardise source columns


In [3]:
# Census TS045
ts045_raw = pd.read_csv(PATHS["ts045"], skiprows=7).rename(columns={
    "Area": "area_raw",
    "No cars or vans in household": "cars_0",
    "1 car or van in household": "cars_1",
    "2 cars or vans in household": "cars_2",
    "3 or more cars or vans in household": "cars_3plus",
})
require_columns(
    ts045_raw,
    ["area_raw", "cars_0", "cars_1", "cars_2", "cars_3plus"],
    "TS045",
)
ts045_raw = ts045_raw.dropna(subset=["area_raw"]).copy()
for col in ["cars_0", "cars_1", "cars_2", "cars_3plus"]:
    ts045_raw[col] = pd.to_numeric(ts045_raw[col], errors="coerce")
ts045_gor_london = ts045_raw.loc[ts045_raw["area_raw"].eq("gor:London")].copy()
ts045_lsoa = ts045_raw.loc[
    ts045_raw["area_raw"].str.startswith("lsoa2021:", na=False)
].copy()
parsed = ts045_lsoa["area_raw"].str.extract(
    r"lsoa2021:(?P<lsoa_code>\S+)\s*:\s*(?P<lsoa_name>.+)"
)
ts045_lsoa[["lsoa_code", "lsoa_name"]] = parsed[["lsoa_code", "lsoa_name"]]
ts045 = ts045_lsoa[
    ["lsoa_code", "lsoa_name", "cars_0", "cars_1", "cars_2", "cars_3plus"]
].reset_index(drop=True)

# IoD2025
iod_raw = pd.read_csv(PATHS["iod2025"])
iod_source_columns = [
    "LSOA code (2021)", "LSOA name (2021)",
    "Local Authority District code (2024)", "Local Authority District name (2024)",
    "Index of Multiple Deprivation (IMD) Score",
    "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)",
    "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)",
    "Income Score (rate)", "Income Rank (where 1 is most deprived)",
    "Income Decile (where 1 is most deprived 10% of LSOAs)",
]
require_columns(iod_raw, iod_source_columns, "IoD2025")
iod2025 = iod_raw[iod_source_columns].rename(columns={
    "LSOA code (2021)": "lsoa_code",
    "LSOA name (2021)": "lsoa_name",
    "Local Authority District code (2024)": "lad_code",
    "Local Authority District name (2024)": "lad_name",
    "Index of Multiple Deprivation (IMD) Score": "imd_score",
    "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)": "imd_rank",
    "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)": "imd_decile",
    "Income Score (rate)": "income_score",
    "Income Rank (where 1 is most deprived)": "income_rank",
    "Income Decile (where 1 is most deprived 10% of LSOAs)": "income_decile",
})

# EVSE registry at connector grain
evse_raw = pd.read_csv(PATHS["evse_location"], low_memory=False)
evse_required = [
    "Source", "id_x", "location_id_x", "location_id_y",
    "address", "city", "postal_code", "state", "location_class",
    "coordinates_latitude_y", "coordinates_longitude",
    "zapmap_device_uid", "power_band", "device_max_power",
    "evse_id", "evse_uid", "id_y", "operator_name",
    "created_at_x", "created_at_y", "created_at", "deleted_at_y",
]
require_columns(evse_raw, evse_required, "evse_location")
evse_connector = evse_raw[evse_required].rename(columns={
    "Source": "source",
    "id_x": "location_id",
    "location_id_x": "location_id_x_source",
    "location_id_y": "location_id_y_source",
    "state": "borough",
    "location_class": "location_category",
    "coordinates_latitude_y": "latitude",
    "coordinates_longitude": "longitude",
    "zapmap_device_uid": "device_id",
    "id_y": "connector_id",
})

# Charging sessions
sessions_raw = pd.read_csv(PATHS["charging_activity"])
require_columns(sessions_raw, ["evse_uid", "start_time", "end_time", "duration"], "charging_activity")
sessions = sessions_raw.rename(columns={
    "start_time": "charging_start",
    "end_time": "charging_end_reported",
    "duration": "charging_duration_min",
})

print(
    f"Loaded: TS045={len(ts045):,}; IoD2025={len(iod2025):,}; "
    f"registry rows={len(evse_connector):,}; sessions={len(sessions):,}"
)


Loaded: TS045=35,672; IoD2025=33,755; registry rows=51,386; sessions=524,454


## 2. Clean Census and deprivation data


In [4]:
# London LAD codes begin E09. Using IoD LAD codes is more robust than parsing borough names.
london_iod = iod2025.loc[
    iod2025["lad_code"].astype(str).str.fullmatch(r"E09\d{6}", na=False)
].copy()
london_codes = set(london_iod["lsoa_code"])

if len(london_codes) != 4_994:
    warnings.warn(f"Expected 4,994 Greater London LSOAs, found {len(london_codes):,}.")

ts045_london = ts045.loc[ts045["lsoa_code"].isin(london_codes)].copy()

band_cols = ["cars_0", "cars_1", "cars_2", "cars_3plus"]
if ts045_london["lsoa_code"].duplicated().any():
    raise ValueError("TS045 contains duplicate London LSOA codes.")
if ts045_london[band_cols].isna().any().any():
    raise ValueError("TS045 contains missing vehicle-band counts in London LSOAs.")
if (ts045_london[band_cols] < 0).any().any():
    raise ValueError("TS045 contains negative vehicle-band counts.")

ts045_london["Hi"] = ts045_london[band_cols].sum(axis=1, min_count=len(band_cols))
ts045_london["vehicle_stock_proxy"] = (
    ts045_london["cars_1"]
    + 2 * ts045_london["cars_2"]
    + TOP_CODED_VEHICLES * ts045_london["cars_3plus"]
)
ts045_london["Ci"] = ts045_london["vehicle_stock_proxy"] / ts045_london["Hi"]

census_london = ts045_london[
    ["lsoa_code", "lsoa_name", "Hi", "Ci", "vehicle_stock_proxy"] + band_cols
].sort_values("lsoa_code").reset_index(drop=True)

if len(census_london) != len(london_codes):
    missing = sorted(london_codes - set(census_london["lsoa_code"]))
    raise ValueError(f"TS045 missing {len(missing)} London LSOAs; examples: {missing[:10]}")

# Independent household-total validation against the Nomis gor:London aggregate.
gor_total = ts045_gor_london[band_cols].sum(axis=1, min_count=len(band_cols))
if gor_total.empty or pd.isna(gor_total.iloc[0]):
    raise ValueError("Could not calculate the gor:London household total from TS045.")
computed_total = census_london["Hi"].sum()
relative_difference = (computed_total - gor_total.iloc[0]) / gor_total.iloc[0]

# IoD London subset and validity checks
iod2025_london = london_iod.sort_values("lsoa_code").reset_index(drop=True)
if iod2025_london["lsoa_code"].duplicated().any():
    raise ValueError("IoD2025 contains duplicate London LSOA codes.")
if set(iod2025_london["lsoa_code"]) != set(census_london["lsoa_code"]):
    raise ValueError("Census and IoD2025 London LSOA code sets do not match.")
if iod2025_london["income_score"].isna().any():
    raise ValueError("IoD2025 income_score contains missing values.")
if not iod2025_london["income_score"].between(0, 1).all():
    raise ValueError("IoD2025 income_score contains values outside [0, 1].")
if not iod2025_london["income_decile"].between(1, 10).all():
    raise ValueError("IoD2025 income_decile contains values outside 1–10.")

census_london.to_csv(OUTPUT_DIR / "census_london_clean.csv", index=False)
iod2025_london.to_csv(OUTPUT_DIR / "imd_london_clean.csv", index=False)

print("Greater London LSOAs:", len(census_london))
print("Computed household total:", f"{computed_total:,.0f}")
print("Nomis gor:London total:", f"{gor_total.iloc[0]:,.0f}")
print("Relative difference:", f"{relative_difference:.4%}")
print("Income-score range:",
      f"{iod2025_london['income_score'].min():.3f}–{iod2025_london['income_score'].max():.3f}")


Greater London LSOAs: 4994
Computed household total: 3,423,845
Nomis gor:London total: 3,423,890
Relative difference: -0.0013%
Income-score range: 0.010–0.998


## 3. Validate the infrastructure hierarchy and create canonical EVSE/location registries

The source contains two related location identifiers. `id_x` / `location_id_x` identify the
location record on the Location–Device side of the joined registry, while `location_id_y`
is the location foreign key stored with the EVSE record. Most rows agree, but a small number
do not. These disagreements are audited rather than treated as an automatic fatal error.

For session-to-location assignment, the EVSE-side `location_id_y` is used because charging
activity is keyed by `evse_uid`. A separate location master table is constructed from `id_x`.
The two are then joined using the selected EVSE location identifier.


In [5]:
# Standardise identifiers and core fields.
id_columns = [
    "location_id", "location_id_x_source", "location_id_y_source",
    "device_id", "evse_uid", "connector_id", "evse_id",
]
for col in id_columns:
    evse_connector[col] = evse_connector[col].astype("string").str.strip()

# Keep the two location concepts explicit.
evse_connector = evse_connector.rename(
    columns={"location_id": "location_record_id"}
)
evse_connector["evse_location_id"] = evse_connector["location_id_y_source"]

evse_connector["location_category"] = (
    evse_connector["location_category"]
    .astype("string")
    .str.strip()
    .str.lower()
)
evse_connector["latitude"] = pd.to_numeric(
    evse_connector["latitude"], errors="coerce"
)
evse_connector["longitude"] = pd.to_numeric(
    evse_connector["longitude"], errors="coerce"
)
evse_connector["device_max_power"] = pd.to_numeric(
    evse_connector["device_max_power"], errors="coerce"
)

for col in ["created_at_x", "created_at_y", "created_at", "deleted_at_y"]:
    evse_connector[col] = pd.to_datetime(
        evse_connector[col], errors="coerce"
    )

required_ids = ["location_record_id", "evse_location_id", "evse_uid"]
if evse_connector[required_ids].isna().any().any():
    missing_counts = evse_connector[required_ids].isna().sum()
    raise ValueError(
        "Registry contains missing structural identifiers:\n"
        f"{missing_counts.to_string()}"
    )

# -------------------------------------------------------------------------
# 3.1 Audit the joined location identifiers
# -------------------------------------------------------------------------
location_x_mismatch = int(
    (~evse_connector["location_record_id"]
      .eq(evse_connector["location_id_x_source"])).sum()
)
location_y_mismatch_mask = (
    ~evse_connector["location_record_id"]
    .eq(evse_connector["evse_location_id"])
)
location_y_mismatch = int(location_y_mismatch_mask.sum())
evse_id_mismatch = int(
    (~evse_connector["evse_id"].eq(evse_connector["evse_uid"])).sum()
)

location_id_disagreements = evse_connector.loc[
    location_y_mismatch_mask,
    [
        "evse_uid", "device_id", "connector_id",
        "location_record_id", "location_id_x_source",
        "evse_location_id", "address", "borough",
        "latitude", "longitude", "created_at_y", "deleted_at_y",
    ],
].sort_values(["evse_uid", "location_record_id"])

location_id_disagreements.to_csv(
    OUTPUT_DIR / "registry_location_id_disagreements.csv",
    index=False,
)

# -------------------------------------------------------------------------
# 3.2 Select one canonical location for each EVSE
# -------------------------------------------------------------------------
# Usually location_id_y is already unique within each EVSE. If not, resolve
# transparently using: most non-deleted rows, then most supporting rows,
# then latest creation timestamp, then lexical ID as a deterministic tie-break.
evse_location_counts = (
    evse_connector.groupby("evse_uid")["evse_location_id"]
    .nunique(dropna=True)
)
evses_with_multiple_direct_locations = evse_location_counts.loc[
    evse_location_counts > 1
].index

mapping_candidates = (
    evse_connector.groupby(
        ["evse_uid", "evse_location_id"],
        as_index=False,
    )
    .agg(
        supporting_rows=("connector_id", "size"),
        non_deleted_rows=("deleted_at_y", lambda s: int(s.isna().sum())),
        latest_created_at=("created_at_y", "max"),
    )
)

mapping_candidates["is_conflicted_evse"] = (
    mapping_candidates["evse_uid"]
    .isin(evses_with_multiple_direct_locations)
)

mapping_candidates = mapping_candidates.sort_values(
    [
        "evse_uid",
        "non_deleted_rows",
        "supporting_rows",
        "latest_created_at",
        "evse_location_id",
    ],
    ascending=[True, False, False, False, True],
    na_position="last",
)

canonical_evse_location = (
    mapping_candidates.drop_duplicates("evse_uid", keep="first")
    [["evse_uid", "evse_location_id"]]
    .rename(columns={"evse_location_id": "location_id"})
)

mapping_candidates.loc[
    mapping_candidates["is_conflicted_evse"]
].to_csv(
    OUTPUT_DIR / "evse_location_mapping_conflicts.csv",
    index=False,
)

if canonical_evse_location["evse_uid"].duplicated().any():
    raise ValueError("Canonical EVSE-location mapping is not one row per EVSE.")

# -------------------------------------------------------------------------
# 3.3 Build a clean location master from the Location-side identifier
# -------------------------------------------------------------------------
# Exclude the small number of mismatched joined rows when constructing the
# primary location attributes, so one location is not contaminated by an
# EVSE record whose own foreign key points elsewhere.
consistent_location_rows = evse_connector.loc[
    evse_connector["location_record_id"]
    .eq(evse_connector["evse_location_id"])
].copy()

def aggregate_location_rows(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby("location_record_id", as_index=False)
        .agg(
            latitude=("latitude", "median"),
            longitude=("longitude", "median"),
            address=("address", mode_or_first),
            city=("city", mode_or_first),
            postal_code=("postal_code", mode_or_first),
            borough=("borough", mode_or_first),
            location_category=("location_category", mode_or_first),
            source=("source", mode_or_first),
            operator_name=("operator_name", mode_or_first),
        )
        .rename(columns={"location_record_id": "location_id"})
    )

location_master = aggregate_location_rows(consistent_location_rows)

# Fallback only for a location that appears exclusively in mismatched rows.
all_location_ids = set(evse_connector["location_record_id"].dropna())
primary_location_ids = set(location_master["location_id"])
missing_location_ids = all_location_ids - primary_location_ids

if missing_location_ids:
    location_fallback = aggregate_location_rows(
        evse_connector.loc[
            evse_connector["location_record_id"].isin(missing_location_ids)
        ]
    )
    location_fallback["location_attribute_source"] = "fallback_all_rows"
    location_master["location_attribute_source"] = "consistent_rows"
    location_master = pd.concat(
        [location_master, location_fallback],
        ignore_index=True,
    )
else:
    location_master["location_attribute_source"] = "consistent_rows"

if location_master["location_id"].duplicated().any():
    raise ValueError("Location master contains duplicate location IDs.")

# Report structural variation instead of failing on harmless coordinate variation.
location_structure_audit = (
    evse_connector.groupby("location_record_id")
    .agg(
        n_latitudes=("latitude", "nunique"),
        n_longitudes=("longitude", "nunique"),
        n_boroughs=("borough", "nunique"),
        n_categories=("location_category", "nunique"),
    )
    .reset_index()
    .rename(columns={"location_record_id": "location_id"})
)
location_structure_audit.to_csv(
    OUTPUT_DIR / "location_structure_audit.csv",
    index=False,
)

# -------------------------------------------------------------------------
# 3.4 Build one row per EVSE and attach canonical location attributes
# -------------------------------------------------------------------------
evse_attributes = (
    evse_connector.groupby("evse_uid", as_index=False)
    .agg(
        device_id=("device_id", mode_or_first),
        source_evse=("source", mode_or_first),
        operator_name_evse=("operator_name", mode_or_first),
        power_band=("power_band", mode_or_first),
        device_max_power=("device_max_power", "max"),
        n_connectors=("connector_id", "nunique"),
        any_deleted_at_y=("deleted_at_y", lambda s: bool(s.notna().any())),
        earliest_created_at=("created_at_y", "min"),
    )
)

evse_registry = (
    canonical_evse_location
    .merge(
        evse_attributes,
        on="evse_uid",
        how="left",
        validate="one_to_one",
    )
    .merge(
        location_master,
        on="location_id",
        how="left",
        validate="many_to_one",
    )
)

# In the unlikely event that a selected EVSE location has no matching Location-side
# row, use the EVSE row's own median coordinates and modal attributes, but flag it.
missing_location_attributes = evse_registry["latitude"].isna()

if missing_location_attributes.any():
    evse_fallback_attributes = (
        evse_connector.groupby("evse_uid", as_index=False)
        .agg(
            latitude_fallback=("latitude", "median"),
            longitude_fallback=("longitude", "median"),
            address_fallback=("address", mode_or_first),
            city_fallback=("city", mode_or_first),
            postal_code_fallback=("postal_code", mode_or_first),
            borough_fallback=("borough", mode_or_first),
            location_category_fallback=("location_category", mode_or_first),
        )
    )
    evse_registry = evse_registry.merge(
        evse_fallback_attributes,
        on="evse_uid",
        how="left",
        validate="one_to_one",
    )
    for col in [
        "latitude", "longitude", "address", "city",
        "postal_code", "borough", "location_category",
    ]:
        evse_registry[col] = evse_registry[col].fillna(
            evse_registry[f"{col}_fallback"]
        )
        evse_registry = evse_registry.drop(
            columns=[f"{col}_fallback"]
        )

evse_registry["used_evse_attribute_fallback"] = missing_location_attributes.to_numpy()

if evse_registry["location_id"].isna().any():
    raise ValueError("Some EVSEs still have no canonical location ID.")
if evse_registry["evse_uid"].duplicated().any():
    raise ValueError("EVSE registry is not one row per EVSE.")

# -------------------------------------------------------------------------
# 3.5 Build one row per location from the canonical EVSE mapping
# -------------------------------------------------------------------------
location_counts = (
    evse_registry.groupby("location_id", as_index=False)
    .agg(
        n_devices=("device_id", "nunique"),
        n_evse=("evse_uid", "nunique"),
        n_connectors=("n_connectors", "sum"),
        max_device_power=("device_max_power", "max"),
        any_deleted_at_y=("any_deleted_at_y", "any"),
    )
)

location_registry = location_master.merge(
    location_counts,
    on="location_id",
    how="inner",
    validate="one_to_one",
)

valid_coords = (
    location_registry["latitude"].between(-90, 90)
    & location_registry["longitude"].between(-180, 180)
)
invalid_coordinate_locations = location_registry.loc[
    ~valid_coords
].copy()
location_registry = location_registry.loc[
    valid_coords
].reset_index(drop=True)

# Keep EVSE registry aligned with the retained valid-location population.
valid_location_ids = set(location_registry["location_id"])
evse_registry = evse_registry.loc[
    evse_registry["location_id"].isin(valid_location_ids)
].reset_index(drop=True)

registry_diagnostics = pd.DataFrame({
    "check": [
        "connector-grain rows",
        "unique Location-side IDs",
        "unique EVSEs",
        "id_x vs location_id_x mismatched rows",
        "id_x vs EVSE-side location_id_y mismatched rows",
        "EVSEs appearing under multiple id_x locations",
        "EVSEs with multiple direct location_id_y values",
        "evse_id vs evse_uid mismatches",
        "locations with invalid coordinates",
        "connector rows with non-null deleted_at_y",
    ],
    "count": [
        len(evse_connector),
        evse_connector["location_record_id"].nunique(),
        evse_connector["evse_uid"].nunique(),
        location_x_mismatch,
        location_y_mismatch,
        int(
            (
                evse_connector.groupby("evse_uid")["location_record_id"]
                .nunique() > 1
            ).sum()
        ),
        len(evses_with_multiple_direct_locations),
        evse_id_mismatch,
        len(invalid_coordinate_locations),
        int(evse_connector["deleted_at_y"].notna().sum()),
    ],
})

registry_diagnostics.to_csv(
    OUTPUT_DIR / "registry_diagnostics.csv",
    index=False,
)
evse_registry.to_csv(
    OUTPUT_DIR / "evse_registry_clean.csv",
    index=False,
)
location_registry.to_csv(
    OUTPUT_DIR / "osev_london_clean.csv",
    index=False,
)

print(registry_diagnostics.to_string(index=False))
print(
    "\nRows with id_x != location_id_y were audited in "
    "'registry_location_id_disagreements.csv'."
)
print(
    "Canonical session-to-location assignment uses the EVSE-side "
    "location_id_y."
)


                                          check  count
                           connector-grain rows  51386
                       unique Location-side IDs  23013
                                   unique EVSEs  38358
          id_x vs location_id_x mismatched rows      0
id_x vs EVSE-side location_id_y mismatched rows     14
  EVSEs appearing under multiple id_x locations     10
EVSEs with multiple direct location_id_y values      0
                 evse_id vs evse_uid mismatches      0
             locations with invalid coordinates      0
      connector rows with non-null deleted_at_y  15652

Rows with id_x != location_id_y were audited in 'registry_location_id_disagreements.csv'.
Canonical session-to-location assignment uses the EVSE-side location_id_y.


## 4. Clean charging sessions and flag potential extremes

The supplied `end_time` is rounded to the minute, while `duration` retains sub-minute
precision. Therefore, the analytical end time is reconstructed as `start + duration`.

Core cleaning removes only records that are mathematically invalid: missing identifiers or
starts, non-positive duration, duration longer than the complete observation window, or an
interval that does not overlap September. Sessions shorter than one minute or longer than 24
hours are flagged for review but retained in the primary analysis.


In [6]:
rows_raw = len(sessions)

sessions["evse_uid"] = sessions["evse_uid"].astype("string").str.strip()
sessions["charging_start"] = pd.to_datetime(sessions["charging_start"], errors="coerce")
sessions["charging_end_reported"] = pd.to_datetime(
    sessions["charging_end_reported"], errors="coerce"
)
sessions["charging_duration_min"] = pd.to_numeric(
    sessions["charging_duration_min"], errors="coerce"
)

# Exact row-level duplicates only. Records sharing a rounded start/end but different duration
# are not treated as duplicates; interval merging handles their overlap later.
duplicate_subset = [
    "evse_uid", "charging_start", "charging_end_reported", "charging_duration_min"
]
n_exact_duplicates = int(sessions.duplicated(subset=duplicate_subset).sum())
sessions = sessions.drop_duplicates(subset=duplicate_subset).copy()

sessions["charging_end_calculated"] = (
    sessions["charging_start"]
    + pd.to_timedelta(sessions["charging_duration_min"], unit="m")
)

invalid_reason = pd.Series(pd.NA, index=sessions.index, dtype="string")
invalid_reason.loc[sessions["evse_uid"].isna() | sessions["evse_uid"].eq("")] = "missing_evse_uid"
invalid_reason.loc[sessions["charging_start"].isna()] = "invalid_start_time"
invalid_reason.loc[sessions["charging_duration_min"].isna()] = "invalid_duration"
invalid_reason.loc[sessions["charging_duration_min"].le(0)] = "non_positive_duration"
invalid_reason.loc[sessions["charging_duration_min"].gt(WINDOW_MINUTES)] = "duration_exceeds_window"
invalid_reason.loc[
    sessions["charging_end_calculated"].le(WINDOW_START)
    | sessions["charging_start"].ge(WINDOW_END)
] = "outside_observation_window"

invalid_sessions = sessions.loc[invalid_reason.notna()].copy()
invalid_sessions["invalid_reason"] = invalid_reason.loc[invalid_reason.notna()]
sessions_clean = sessions.loc[invalid_reason.isna()].copy()

sessions_clean["flag_short_lt_1min"] = (
    sessions_clean["charging_duration_min"] < SHORT_SESSION_THRESHOLD_MIN
)
sessions_clean["flag_long_gt_24h"] = (
    sessions_clean["charging_duration_min"] > LONG_SESSION_THRESHOLD_MIN
)

# Registry join must not fan out because evse_registry is one row per evse_uid.
active_not_in_registry = sorted(
    set(sessions_clean["evse_uid"]) - set(evse_registry["evse_uid"])
)
if active_not_in_registry:
    raise ValueError(
        f"{len(active_not_in_registry)} active EVSEs are absent from the registry; "
        f"examples: {active_not_in_registry[:10]}"
    )

sessions_clean = sessions_clean.merge(
    evse_registry[
        ["evse_uid", "location_id", "latitude", "longitude", "borough",
         "location_category", "power_band", "device_max_power"]
    ],
    on="evse_uid",
    how="left",
    validate="many_to_one",
)
sessions_clean["hour"] = sessions_clean["charging_start"].dt.hour
sessions_clean["dayofweek"] = sessions_clean["charging_start"].dt.dayofweek

session_cleaning_summary = pd.DataFrame({
    "item": [
        "raw session rows",
        "exact duplicate rows removed",
        "invalid rows removed",
        "clean rows retained",
        "retained sessions shorter than 1 minute",
        "retained sessions longer than 24 hours",
        "unique active EVSEs",
    ],
    "count": [
        rows_raw,
        n_exact_duplicates,
        len(invalid_sessions),
        len(sessions_clean),
        int(sessions_clean["flag_short_lt_1min"].sum()),
        int(sessions_clean["flag_long_gt_24h"].sum()),
        sessions_clean["evse_uid"].nunique(),
    ],
})

sessions_clean.to_csv(OUTPUT_DIR / "zapmap_clean.csv", index=False)
invalid_sessions.to_csv(OUTPUT_DIR / "charging_sessions_invalid.csv", index=False)
session_cleaning_summary.to_csv(OUTPUT_DIR / "session_cleaning_summary.csv", index=False)

print(session_cleaning_summary.to_string(index=False))
print("\nDuration quantiles (minutes):")
print(sessions_clean["charging_duration_min"].quantile(
    [0, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999, 1]
))


                                   item  count
                       raw session rows 524454
           exact duplicate rows removed    338
                   invalid rows removed      0
                    clean rows retained 524116
retained sessions shorter than 1 minute  58490
 retained sessions longer than 24 hours      0
                    unique active EVSEs  18748

Duration quantiles (minutes):
0.000       0.016667
0.010       0.033333
0.050       0.300000
0.500      32.433333
0.950     350.033333
0.990     658.695000
0.999     961.911667
1.000    1381.400000
Name: charging_duration_min, dtype: float64


## 5. Compute utilisation for all registered EVSEs, including zero-session EVSEs


In [7]:
occupied_records = []
for evse_uid, group in sessions_clean.groupby("evse_uid", sort=False):
    occupied_minutes = merge_intervals_minutes(
        group["charging_start"],
        group["charging_end_calculated"],
        WINDOW_START,
        WINDOW_END,
    )
    occupied_records.append((evse_uid, occupied_minutes, len(group)))

active_utilisation = pd.DataFrame(
    occupied_records,
    columns=["evse_uid", "occupied_minutes", "n_session_records"],
)
active_utilisation["ur_j"] = active_utilisation["occupied_minutes"] / WINDOW_MINUTES

# Left join from the complete registry so EVSEs with no observed session receive ur_j = 0.
evse_utilisation_all = evse_registry.merge(
    active_utilisation,
    on="evse_uid",
    how="left",
    validate="one_to_one",
)
evse_utilisation_all["occupied_minutes"] = (
    evse_utilisation_all["occupied_minutes"].fillna(0.0)
)
evse_utilisation_all["n_session_records"] = (
    evse_utilisation_all["n_session_records"].fillna(0).astype(int)
)
evse_utilisation_all["ur_j"] = evse_utilisation_all["ur_j"].fillna(0.0)
evse_utilisation_all["has_session"] = evse_utilisation_all["n_session_records"].gt(0)

if not evse_utilisation_all["ur_j"].between(0, 1).all():
    bad = evse_utilisation_all.loc[
        ~evse_utilisation_all["ur_j"].between(0, 1),
        ["evse_uid", "occupied_minutes", "ur_j"],
    ]
    raise ValueError(f"EVSE utilisation outside [0,1]:\n{bad.head()}")

# This compatibility filename now deliberately contains every registry EVSE, not only active EVSEs.
evse_utilisation_all.to_csv(OUTPUT_DIR / "evse_ur_clean.csv", index=False)
evse_utilisation_all.to_csv(OUTPUT_DIR / "evse_utilisation_all.csv", index=False)

print("Registered EVSEs:", f"{len(evse_utilisation_all):,}")
print("EVSEs with at least one session:", f"{evse_utilisation_all['has_session'].sum():,}")
print("Zero-session EVSEs assigned ur_j=0:",
      f"{(~evse_utilisation_all['has_session']).sum():,}")
print("\nur_j distribution across all registered EVSEs:")
print(evse_utilisation_all["ur_j"].describe())


Registered EVSEs: 38,358
EVSEs with at least one session: 18,748
Zero-session EVSEs assigned ur_j=0: 19,610

ur_j distribution across all registered EVSEs:
count    38358.000000
mean         0.022964
std          0.056350
min          0.000000
25%          0.000000
50%          0.000000
75%          0.020426
max          0.701196
Name: ur_j, dtype: float64


## 6. Aggregate EVSE utilisation to location level

For location `l`:

- `ubar_loc` is the mean `ur_j` across **all** EVSEs registered at the location;
- zero-session EVSEs therefore remain in the denominator; and
- `seff_loc` is set equal to `ubar_loc`, so each location contributes between 0 and 1
  location-equivalent units of effective supply.

`seff_evse_raw` is retained only as a diagnostic. It is not used in the location-based model.


In [8]:
evse_location_agg = (
    evse_utilisation_all.groupby("location_id", as_index=False)
    .agg(
        n_evse_from_utilisation=("evse_uid", "nunique"),
        n_active_evse=("has_session", "sum"),
        ubar_loc=("ur_j", "mean"),
        seff_evse_raw=("ur_j", "sum"),
        occupied_minutes_total=("occupied_minutes", "sum"),
    )
)

location_utilisation = location_registry.merge(
    evse_location_agg,
    on="location_id",
    how="left",
    validate="one_to_one",
)

# A location in the registry must contain at least one registry EVSE.
if location_utilisation["ubar_loc"].isna().any():
    raise ValueError("Some registered locations have no EVSE rows after aggregation.")
if not (location_utilisation["n_evse"] == location_utilisation["n_evse_from_utilisation"]).all():
    raise ValueError("Location-level EVSE counts do not match between registry and utilisation tables.")

location_utilisation["n_active_evse"] = location_utilisation["n_active_evse"].astype(int)
location_utilisation["has_session"] = location_utilisation["n_active_evse"].gt(0)
location_utilisation["seff_loc"] = location_utilisation["ubar_loc"]

if not location_utilisation["ubar_loc"].between(0, 1).all():
    raise ValueError("Location mean utilisation contains values outside [0,1].")

location_utilisation.to_csv(OUTPUT_DIR / "location_utilisation_clean.csv", index=False)

print("Locations:", f"{len(location_utilisation):,}")
print("Locations with at least one active EVSE:",
      f"{location_utilisation['has_session'].sum():,}")
print("Zero-session locations:", f"{(~location_utilisation['has_session']).sum():,}")
print("\nLocation-level utilisation:")
print(location_utilisation[["ubar_loc", "seff_evse_raw"]].describe())


Locations: 23,008
Locations with at least one active EVSE: 14,460
Zero-session locations: 8,548

Location-level utilisation:
           ubar_loc  seff_evse_raw
count  23008.000000   23008.000000
mean       0.021317       0.038284
std        0.039527       0.177278
min        0.000000       0.000000
25%        0.000000       0.000000
50%        0.004600       0.004976
75%        0.025263       0.030190
max        0.552668       7.365296


## 7. Spatially assign locations to 2021 LSOAs


In [9]:
lsoa_boundaries = gpd.read_file(PATHS["lsoa_boundaries"])
require_columns(lsoa_boundaries, ["LSOA21CD", "geometry"], "LSOA boundaries")

if lsoa_boundaries.crs is None:
    raise ValueError("LSOA boundary file has no CRS.")
lsoa_boundaries = lsoa_boundaries.to_crs(epsg=27700)

lsoa_london = (
    lsoa_boundaries.loc[lsoa_boundaries["LSOA21CD"].isin(london_codes), ["LSOA21CD", "geometry"]]
    .rename(columns={"LSOA21CD": "lsoa_code"})
    .copy()
)
if lsoa_london["lsoa_code"].duplicated().any():
    raise ValueError("LSOA boundary file contains duplicate London LSOA codes.")
if len(lsoa_london) != len(london_codes):
    raise ValueError(
        f"Boundary match incomplete: {len(lsoa_london):,} / {len(london_codes):,} London LSOAs."
    )

location_points = gpd.GeoDataFrame(
    location_utilisation.copy(),
    geometry=gpd.points_from_xy(
        location_utilisation["longitude"], location_utilisation["latitude"]
    ),
    crs="EPSG:4326",
).to_crs(epsg=27700)

location_joined = gpd.sjoin(
    location_points,
    lsoa_london,
    how="left",
    predicate="within",
).drop(columns=["index_right"])

if location_joined.index.duplicated().any():
    raise ValueError("A location matched more than one LSOA in the within join.")

location_joined["match_method"] = np.where(
    location_joined["lsoa_code"].notna(), "within", pd.NA
)
location_joined["nearest_distance_m"] = np.nan

# Resolve boundary-edge points with a tightly bounded nearest join.
unmatched_index = location_joined.index[location_joined["lsoa_code"].isna()]
if len(unmatched_index):
    nearest_input = location_points.loc[unmatched_index].drop(columns=[], errors="ignore")
    nearest = gpd.sjoin_nearest(
        nearest_input,
        lsoa_london,
        how="left",
        max_distance=NEAREST_MATCH_MAX_METRES,
        distance_col="nearest_distance_m",
    )
    if nearest.index.duplicated().any():
        nearest = (
            nearest.sort_values("nearest_distance_m")
            .loc[~nearest.index.duplicated(keep="first")]
        )
    resolved = nearest["lsoa_code"].notna()
    resolved_index = nearest.index[resolved]
    location_joined.loc[resolved_index, "lsoa_code"] = nearest.loc[resolved_index, "lsoa_code"]
    location_joined.loc[resolved_index, "match_method"] = "nearest"
    location_joined.loc[resolved_index, "nearest_distance_m"] = nearest.loc[
        resolved_index, "nearest_distance_m"
    ]

location_lsoa = pd.DataFrame(location_joined.drop(columns="geometry"))
unmatched_locations = location_lsoa.loc[location_lsoa["lsoa_code"].isna()].copy()

location_lsoa.to_csv(OUTPUT_DIR / "location_lsoa_clean.csv", index=False)
unmatched_locations.to_csv(OUTPUT_DIR / "location_lsoa_unmatched.csv", index=False)

print("London LSOA boundaries:", len(lsoa_london))
print("Locations matched within polygon:",
      f"{location_lsoa['match_method'].eq('within').sum():,}")
print("Locations matched by nearest boundary correction:",
      f"{location_lsoa['match_method'].eq('nearest').sum():,}")
print("Locations still unmatched:", f"{len(unmatched_locations):,}")


London LSOA boundaries: 4994
Locations matched within polygon: 23,007
Locations matched by nearest boundary correction: 1
Locations still unmatched: 0


## 8. Construct location-consistent LSOA supply variables

Only registered on-street locations are used for the existing-provision variables.

For LSOA `i`:

- `e_i` = number of on-street charging locations;
- `U_i` = mean location utilisation among those locations;
- `S_eff_i` = sum of location mean utilisation; and
- `has_evse_i` = whether at least one on-street charging location is registered.

By construction, `S_eff_i = e_i × U_i` whenever `e_i > 0`, and `0 ≤ S_eff_i ≤ e_i`.


In [10]:
onstreet_locations = location_lsoa.loc[
    location_lsoa["location_category"].eq("on-street")
    & location_lsoa["lsoa_code"].notna()
].copy()

lsoa_supply_observed = (
    onstreet_locations.groupby("lsoa_code", as_index=False)
    .agg(
        e_i=("location_id", "nunique"),
        S_eff_i=("seff_loc", "sum"),
        U_i=("ubar_loc", "mean"),
        n_evse=("n_evse", "sum"),
        n_active_evse=("n_active_evse", "sum"),
        n_active_locations=("has_session", "sum"),
    )
)

lsoa_supply = census_london[["lsoa_code", "lsoa_name"]].merge(
    lsoa_supply_observed,
    on="lsoa_code",
    how="left",
    validate="one_to_one",
)

lsoa_supply["e_i"] = lsoa_supply["e_i"].fillna(0).astype(int)
lsoa_supply["S_eff_i"] = lsoa_supply["S_eff_i"].fillna(0.0)
lsoa_supply["n_evse"] = lsoa_supply["n_evse"].fillna(0).astype(int)
lsoa_supply["n_active_evse"] = lsoa_supply["n_active_evse"].fillna(0).astype(int)
lsoa_supply["n_active_locations"] = (
    lsoa_supply["n_active_locations"].fillna(0).astype(int)
)
lsoa_supply["has_evse_i"] = lsoa_supply["e_i"].gt(0)
# U_i remains NaN where no location exists; a neutral treatment rule belongs in methodology.

if (lsoa_supply["S_eff_i"] > lsoa_supply["e_i"] + 1e-10).any():
    raise ValueError("Location-equivalent effective supply exceeds the location count.")

with_supply = lsoa_supply.loc[lsoa_supply["has_evse_i"]].copy()
identity_error = (
    with_supply["S_eff_i"] - with_supply["e_i"] * with_supply["U_i"]
).abs().max()
if identity_error > 1e-10:
    raise ValueError(f"S_eff_i != e_i * U_i; maximum error={identity_error}")

lsoa_supply.to_csv(OUTPUT_DIR / "lsoa_supply_clean.csv", index=False)
# Compatibility name for downstream work; columns now use explicit location-consistent names.
lsoa_supply.to_csv(OUTPUT_DIR / "seff_london.csv", index=False)

print("On-street locations included:", f"{len(onstreet_locations):,}")
print("LSOAs with at least one on-street location:",
      f"{lsoa_supply['has_evse_i'].sum():,}")
print("LSOAs with zero on-street locations:",
      f"{(~lsoa_supply['has_evse_i']).sum():,}")
print("Maximum e_i:", int(lsoa_supply["e_i"].max()))
print("Maximum identity error |S_eff_i - e_i*U_i|:", identity_error)
print("\nLSOA supply summary:")
print(lsoa_supply[["e_i", "S_eff_i", "U_i"]].describe())


On-street locations included: 21,364
LSOAs with at least one on-street location: 3,157
LSOAs with zero on-street locations: 1,837
Maximum e_i: 63
Maximum identity error |S_eff_i - e_i*U_i|: 2.220446049250313e-16

LSOA supply summary:
               e_i      S_eff_i          U_i
count  4994.000000  4994.000000  3157.000000
mean      4.277934     0.087880     0.026427
std       7.046983     0.162225     0.033768
min       0.000000     0.000000     0.000000
25%       0.000000     0.000000     0.005545
50%       1.000000     0.012959     0.016173
75%       5.000000     0.111208     0.034569
max      63.000000     1.758645     0.386982


## 9. Final audit and output summary


In [11]:
output_summary = pd.DataFrame({
    "output": [
        "census_london_clean.csv",
        "imd_london_clean.csv",
        "evse_registry_clean.csv",
        "osev_london_clean.csv",
        "zapmap_clean.csv",
        "evse_utilisation_all.csv",
        "location_utilisation_clean.csv",
        "location_lsoa_clean.csv",
        "lsoa_supply_clean.csv",
    ],
    "rows": [
        len(census_london),
        len(iod2025_london),
        len(evse_registry),
        len(location_registry),
        len(sessions_clean),
        len(evse_utilisation_all),
        len(location_utilisation),
        len(location_lsoa),
        len(lsoa_supply),
    ],
    "observation_unit": [
        "LSOA", "LSOA", "EVSE", "charging location", "charging session",
        "EVSE", "charging location", "charging location", "LSOA",
    ],
})
output_summary.to_csv(OUTPUT_DIR / "pipeline_summary.csv", index=False)

checks = {
    "Census and IoD code sets match": set(census_london["lsoa_code"]) == set(iod2025_london["lsoa_code"]),
    "All active EVSEs found in registry": len(active_not_in_registry) == 0,
    "All EVSE utilisation values in [0,1]": evse_utilisation_all["ur_j"].between(0, 1).all(),
    "All location utilisation values in [0,1]": location_utilisation["ubar_loc"].between(0, 1).all(),
    "S_eff_i never exceeds e_i": (lsoa_supply["S_eff_i"] <= lsoa_supply["e_i"] + 1e-10).all(),
    "LSOA supply has one row per London LSOA": len(lsoa_supply) == len(census_london),
}

print(output_summary.to_string(index=False))
print("\nFinal checks:")
for label, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {label}")

if not all(checks.values()):
    raise AssertionError("One or more final cleaning checks failed.")


                        output   rows  observation_unit
       census_london_clean.csv   4994              LSOA
          imd_london_clean.csv   4994              LSOA
       evse_registry_clean.csv  38358              EVSE
         osev_london_clean.csv  23008 charging location
              zapmap_clean.csv 524116  charging session
      evse_utilisation_all.csv  38358              EVSE
location_utilisation_clean.csv  23008 charging location
       location_lsoa_clean.csv  23008 charging location
         lsoa_supply_clean.csv   4994              LSOA

Final checks:
PASS — Census and IoD code sets match
PASS — All active EVSEs found in registry
PASS — All EVSE utilisation values in [0,1]
PASS — All location utilisation values in [0,1]
PASS — S_eff_i never exceeds e_i
PASS — LSOA supply has one row per London LSOA


## Notes before rerunning EDA

The previous `03_EDA.ipynb` must not reconstruct `ubar_loc` or `S_eff_i` from active EVSEs.
It should instead load:

- `location_lsoa_clean.csv` for location-level maps and diagnostics; and
- `lsoa_supply_clean.csv` for `e_i`, `U_i`, `S_eff_i` and `has_evse_i`.

This prevents zero-session EVSEs from disappearing from the utilisation denominator and keeps
all supply variables in the charging-location unit.


In [12]:
# Diagnostic: identify Location-side IDs that never became any EVSE's canonical location
all_location_ids = set(location_master["location_id"])
retained_location_ids = set(location_counts["location_id"])
orphaned_location_ids = all_location_ids - retained_location_ids

print(f"Orphaned Location-side IDs (in location_master but never canonical): {len(orphaned_location_ids)}")

orphaned_detail = location_master.loc[
    location_master["location_id"].isin(orphaned_location_ids)
]
display(orphaned_detail[[
    "location_id", "address", "borough", "location_category",
    "location_attribute_source", "latitude", "longitude"
]])

# Check whether these locations ever had an EVSE pointing at them via id_x,
# even if that EVSE's canonical location was resolved elsewhere
rows_touching_orphans = evse_connector.loc[
    evse_connector["location_record_id"].isin(orphaned_location_ids)
]
print("\nConnector-grain rows referencing these locations via id_x:", len(rows_touching_orphans))
display(rows_touching_orphans[[
    "evse_uid", "location_record_id", "location_id_x_source",
    "evse_location_id", "address", "borough", "deleted_at_y"
]].sort_values("location_record_id"))

orphaned_detail.to_csv(OUTPUT_DIR / "orphaned_locations.csv", index=False)

Orphaned Location-side IDs (in location_master but never canonical): 5


,location_id,address,borough,location_category,location_attribute_source,latitude,longitude
23008,AS72JSH,Richmond Road 383,Richmond upon Thames,destination,fallback_all_rows,51.455903,-0.310271
23009,BAJMSSV,33 Horseferry Road,Westminster,on-street,fallback_all_rows,51.494778,-0.130215
23010,DGJVIPV,Eastern Avenue West,Havering,destination,fallback_all_rows,51.582849,0.167280
23011,JLN6DKW,6 AMOR ROAD,Hammersmith and Fulham,on-street,fallback_all_rows,51.498219,-0.228258
23012,LNSJ7YT,Amor Road,Hammersmith and Fulham,on-street,fallback_all_rows,51.498219,-0.228258



Connector-grain rows referencing these locations via id_x: 13


,evse_uid,location_record_id,location_id_x_source,evse_location_id,address,borough,deleted_at_y
34582,1c5f2d35f902cc1d2954da3f93f3aecd,AS72JSH,AS72JSH,3PGKIPZ,Richmond Road 383,Richmond upon Thames,NaT
34583,d114c868f8c7f831b613403a86e25651,AS72JSH,AS72JSH,3PGKIPZ,Richmond Road 383,Richmond upon Thames,2025-09-08 15:21:08
34584,6bafb66e13c2aadc322c60b5c6c40f20,AS72JSH,AS72JSH,3PGKIPZ,Richmond Road 383,Richmond upon Thames,2025-09-08 15:21:08
47809,9367875122714672982e42f189c1d63f,BAJMSSV,BAJMSSV,B2BOYKX,33 Horseferry Road,Westminster,NaT
47810,9367875122714672982e42f189c1d63f,BAJMSSV,BAJMSSV,B2BOYKX,33 Horseferry Road,Westminster,2024-11-26 14:33:39
47811,9367875122714672982e42f189c1d63f,BAJMSSV,BAJMSSV,B2BOYKX,33 Horseferry Road,Westminster,NaT
47812,e5f4cd1a04264e59afb5a81251757018,BAJMSSV,BAJMSSV,B2BOYKX,33 Horseferry Road,Westminster,2024-11-26 14:33:39
47813,e5f4cd1a04264e59afb5a81251757018,BAJMSSV,BAJMSSV,B2BOYKX,33 Horseferry Road,Westminster,NaT
47814,e5f4cd1a04264e59afb5a81251757018,BAJMSSV,BAJMSSV,B2BOYKX,33 Horseferry Road,Westminster,NaT
21058,876e2828fa6d4df46c92324da2473cae,DGJVIPV,DGJVIPV,2S5DK0F,Eastern Avenue West,Havering,NaT


In [13]:
# Sensitivity check: does excluding sub-1-minute sessions materially change utilisation?
def recompute_occupied_minutes(sessions_subset):
    records = []
    for evse_uid, group in sessions_subset.groupby("evse_uid", sort=False):
        occ = merge_intervals_minutes(
            group["charging_start"], group["charging_end_calculated"],
            WINDOW_START, WINDOW_END,
        )
        records.append((evse_uid, occ))
    return pd.DataFrame(records, columns=["evse_uid", "occupied_minutes_excl_short"])

sessions_excl_short = sessions_clean.loc[~sessions_clean["flag_short_lt_1min"]]
ur_excl_short = recompute_occupied_minutes(sessions_excl_short)
ur_excl_short["ur_j_excl_short"] = ur_excl_short["occupied_minutes_excl_short"] / WINDOW_MINUTES

comparison = evse_utilisation_all[["evse_uid", "location_id", "ur_j"]].merge(
    ur_excl_short[["evse_uid", "ur_j_excl_short"]], on="evse_uid", how="left"
)
comparison["ur_j_excl_short"] = comparison["ur_j_excl_short"].fillna(0.0)
comparison["abs_diff"] = (comparison["ur_j"] - comparison["ur_j_excl_short"]).abs()

print("EVSE-level ur_j difference (with vs without sub-1-min sessions):")
print(comparison["abs_diff"].describe())
print("Pearson correlation:", comparison["ur_j"].corr(comparison["ur_j_excl_short"]))

# Aggregate to LSOA-level U_i under the exclusion scenario and compare
loc_excl = comparison.groupby("location_id", as_index=False)["ur_j_excl_short"].mean() \
    .rename(columns={"ur_j_excl_short": "ubar_loc_excl_short"})
supply_check = location_lsoa[["location_id", "lsoa_code"]].merge(loc_excl, on="location_id", how="left")
Ui_excl = supply_check.groupby("lsoa_code", as_index=False)["ubar_loc_excl_short"].mean()
Ui_compare = lsoa_supply[["lsoa_code", "U_i"]].merge(Ui_excl, on="lsoa_code", how="left")
print("\nLSOA-level U_i correlation (all sessions vs excl. <1min):",
      Ui_compare["U_i"].corr(Ui_compare["ubar_loc_excl_short"]))

EVSE-level ur_j difference (with vs without sub-1-min sessions):
count    38358.000000
mean         0.000012
std          0.000083
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.002967
Name: abs_diff, dtype: float64
Pearson correlation: 0.9999990081430252

LSOA-level U_i correlation (all sessions vs excl. <1min): 0.9490763141132122


In [16]:
onstreet_location_ids = set(onstreet_locations["location_id"])
loc_excl_onstreet = loc_excl.loc[loc_excl["location_id"].isin(onstreet_location_ids)]
supply_check_onstreet = onstreet_locations[["location_id", "lsoa_code"]].merge(
    loc_excl_onstreet, on="location_id", how="left"
)
Ui_excl_onstreet = supply_check_onstreet.groupby("lsoa_code", as_index=False)["ubar_loc_excl_short"].mean()
Ui_compare_onstreet = lsoa_supply[["lsoa_code", "U_i"]].merge(Ui_excl_onstreet, on="lsoa_code", how="left")
print("LSOA-level U_i correlation (on-street only, all sessions vs excl. <1min):",
      Ui_compare_onstreet["U_i"].corr(Ui_compare_onstreet["ubar_loc_excl_short"]))

LSOA-level U_i correlation (on-street only, all sessions vs excl. <1min): 0.9999989484051938


In [14]:
top_lsoa_code = lsoa_supply.loc[lsoa_supply["e_i"].idxmax(), "lsoa_code"]
top_lsoa_locations = onstreet_locations.loc[onstreet_locations["lsoa_code"] == top_lsoa_code]

print(f"LSOA with maximum e_i: {top_lsoa_code}")
print("Unique location_id:", top_lsoa_locations["location_id"].nunique())
print("Unique coordinate pairs:", top_lsoa_locations[["latitude", "longitude"]].drop_duplicates().shape[0])
display(
    top_lsoa_locations[["location_id", "address", "latitude", "longitude", "borough"]]
    .sort_values("address")
)

LSOA with maximum e_i: E01004232
Unique location_id: 63
Unique coordinate pairs: 62


,location_id,address,latitude,longitude,borough
12999,EAIVKPV,1 Aberavon Road,51.525533,-0.034261,Tower Hamlets
18924,QNAIGLD,11A Tredegar Square,51.527148,-0.031292,Tower Hamlets
11026,CEDJIUQ,14 Morgan Street,51.527757,-0.031040,Tower Hamlets
10072,BH7HCS3,17 Aberavon Road,51.525973,-0.034628,Tower Hamlets
1972,2T0NRUZ,18 Lichfield Road,51.527863,-0.035521,Tower Hamlets
...,...,...,...,...,...
6210,7FALGU9,Outside 83 Clinton Road,51.526750,-0.037104,Tower Hamlets
5672,6UBTSQ1,Outside 9-10 College Terrace,51.527735,-0.033212,Tower Hamlets
3923,4WYF2HL,Outside 90 Lichfield Road,51.527672,-0.035953,Tower Hamlets
6140,7CM1WFC,Outside 92 Clinton Road,51.526862,-0.037120,Tower Hamlets


In [15]:
# Empirical test of what deleted_at_y actually means, without relying on documentation

# 1. Does deletion status correlate with having any session at all?
deletion_vs_activity = pd.crosstab(
    evse_utilisation_all["any_deleted_at_y"],
    evse_utilisation_all["has_session"],
    margins=True
)
print("any_deleted_at_y vs has_session:")
print(deletion_vs_activity)

# 2. For EVSEs flagged as deleted but that DO have September sessions,
#    do those sessions fall before or after the deletion timestamp?
deleted_evse_ids = evse_connector.loc[
    evse_connector["deleted_at_y"].notna(), "evse_uid"
].unique()

deleted_with_sessions = sessions_clean.loc[
    sessions_clean["evse_uid"].isin(deleted_evse_ids)
].merge(
    evse_connector.groupby("evse_uid", as_index=False)["deleted_at_y"].max(),
    on="evse_uid", how="left"
)
deleted_with_sessions["session_after_deletion"] = (
    deleted_with_sessions["charging_start"] > deleted_with_sessions["deleted_at_y"]
)

print(f"\nEVSEs flagged deleted_at_y that still have Sept sessions: "
      f"{deleted_with_sessions['evse_uid'].nunique()} / {len(deleted_evse_ids)}")
print("Of their sessions, how many start AFTER the recorded deleted_at_y timestamp:")
print(deleted_with_sessions["session_after_deletion"].value_counts())

# 3. Are the 19,610 zero-session EVSEs disproportionately the ones flagged deleted_at_y?
zero_session = evse_utilisation_all.loc[~evse_utilisation_all["has_session"]]
print(f"\nZero-session EVSEs with any_deleted_at_y=True: "
      f"{zero_session['any_deleted_at_y'].sum():,} / {len(zero_session):,} "
      f"({zero_session['any_deleted_at_y'].mean():.1%})")

active_session = evse_utilisation_all.loc[evse_utilisation_all["has_session"]]
print(f"Active EVSEs with any_deleted_at_y=True: "
      f"{active_session['any_deleted_at_y'].sum():,} / {len(active_session):,} "
      f"({active_session['any_deleted_at_y'].mean():.1%})")

any_deleted_at_y vs has_session:
has_session       False   True    All
any_deleted_at_y                     
False             10899  15956  26855
True               8711   2792  11503
All               19610  18748  38358

EVSEs flagged deleted_at_y that still have Sept sessions: 2792 / 11503
Of their sessions, how many start AFTER the recorded deleted_at_y timestamp:
session_after_deletion
True     83315
False     7344
Name: count, dtype: int64

Zero-session EVSEs with any_deleted_at_y=True: 8,711 / 19,610 (44.4%)
Active EVSEs with any_deleted_at_y=True: 2,792 / 18,748 (14.9%)
